### Stacking algorithm - regression example

In [13]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import tensorflow as tf
import keras
from keras import layers


In [14]:
# load the diamonds dataset, replace this with your own .csv -data
# this version of data should be READY TO GO to training
# which means => only numeric values in the dataset
df = pd.read_csv("processed_regression_diamonds.csv")
df.head()

,carat,cut,color,clarity,table,price,x,y,z
0,0.29,4,2,4,58.0,334,4.20,4.23,2.63
1,0.26,3,3,3,55.0,337,4.07,4.11,2.53
2,0.32,4,6,1,58.0,345,4.38,4.42,2.68
3,0.30,5,2,2,54.0,348,4.31,4.34,2.68
4,0.30,3,1,3,59.0,351,4.21,4.27,2.66


In [15]:
# CatBoost requires we save our continuous and categorical variables separately into lists
categorical_variables = ["cut", "color", "clarity"]

# continuous variables also into a list
continuous_variables = ["carat", "table", "price", "depth", "x", "y", "z"]

# the usual X/y -split
X = df.drop("price", axis=1)
y = df['price']

# usual train/test -split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# NOTE! SCALING => some of the algorithms require this
scaler = StandardScaler()

# create separate versions for the scaled data
# because we need both unscaled and scaled versions later
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Stacking algorithms - regression example

In [16]:
from sklearn.ensemble import StackingRegressor

# meta model is the "judge" or the optimizer of the stacking algorithm
# => it assigns weights to the models based on their performance
meta_model = LinearRegression()

# stacking regression algorithm
stacking_model = StackingRegressor(
    estimators=[('xgb', xgb.XGBRegressor(enable_categorical=True, objective="reg:squarederror")),
                ('randomforest', RandomForestRegressor()),
                ('lgbm', lgb.LGBMRegressor(objective="regression"))],
    final_estimator=meta_model,
    cv=5
)

# other than that, the stacking algorithm works as any other algorithm
stacking_model.fit(X_train, y_train)

# test predictions and metrics
predictions = stacking_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R2: {r2}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000183 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 923
[LightGBM] [Info] Number of data points in the train set: 33580, number of used features: 8
[LightGBM] [Info] Start training from score 2329.642853
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000227 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 913
[LightGBM] [Info] Number of data points in the train set: 26864, number of used features: 8
[LightGBM] [Info] Start training from score 2327.950826
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000110 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

In [ ]:
# pretty decent result
# original MAE with Random Forest => 141.9
# original RMSE with CatBoost => 232.67
# original R2 with CatBoost => 0.9825